In [10]:
import re

# Features to delete (set for O(1) lookups)
FEATURES_TO_DELETE = {
    "mailing_address_city", "receiver_address_line2", "last_updated_date",
    "unstructured_addenda_info_raw", "mailing_address_country_code", "deceased",
    "sender_address_line2", "mailing_address_postal_code", "work_phone_verified",
    "home_phone_verified_date", "common_name_prefix", "unstructured_addenda_info",
    "home_phone_number", "receiver_address_country", "fed_ack_time", "created_by",
    "mobile_phone_verified_date", "mailing_address_line1", "legal_first_name",
    "type_sub_type", "instructing_bank_name", "work_phone_area_code",
    "receiver_customer_status_description", "home_phone_designation", "sofi_employee",
    "home_phone_sms_accepted_ind", "legacy_fraud_type", "work_phone_number",
    "old_two_factor_type", "mailing_address_state_name", "mobile_phone_verified",
    "local_instrument_code", "mobile_phone_extension", "currency", "customer_gender",
    "ofac_ticket_action", "work_phone_verified_date", "asset_movement_status",
    "created_for", "legacy_fraud_status", "external_ref_number", "home_phone_area_code",
    "old_two_factor_data", "mobile_phone_sms_accepted_ind", "mailing_address_line2",
    "mobile_phone_country_prefix", "drivers_license_number", "wire_facilitator",
    "receiver_risk_group", "fee_movement_id", "common_middle_name", "fee_transaction_id",
    "home_phone_country_prefix", "fee_amount", "file_id", "movement_id", "home_phone_verified",
    "business_function_code", "home_address_unit_no", "previous_imad", "created_date",
    "receiver_available_balance", "legal_last_name", "wire_transaction_id",
    "mailing_address_state_code", "is_in_fraud_event_id", "instructing_bank_address",
    "business_unit", "is_in_fraud_user"
}

import json

def clean_event_type_json(input_path, output_path, features_to_delete):
    """
    Reads 'event_type.json', deletes any dfeSchemaEntries or eventAttributeEntries corresponding
    to features in features_to_delete, writes updated json to output_path, and returns
    info about which features were processed and count of rows deleted.
    Now handles JSON decoding errors gracefully with a helpful message.
    """
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        print(
            "\n[ERROR] Could not load JSON due to a decoding error.\n"
            "Details: {}\n"
            "Check if the JSON file is valid -- possibly keys/strings are using single quotes\n"
            "or there are trailing commas or comments (which are not allowed).\n"
            "Line {}, column {} (char {})\n".format(
                e.msg, e.lineno, e.colno, e.pos
            )
        )
        # Optionally, re-raise or return here
        return

    count_deleted = 0
    processed_features = set()

    # Clean dfeSchemaEntries
    try:
        entries = data["clients"]["1000"]["eventConfig"]["dfeSchemaEntries"]
        kept_entries = []
        for entry in entries:
            name = entry.get("dfeName")
            if name and name in features_to_delete:
                count_deleted += 1
                processed_features.add(name)
                continue
            kept_entries.append(entry)
        data["clients"]["1000"]["eventConfig"]["dfeSchemaEntries"] = kept_entries
    except Exception as e:
        print(f"Failed dfeSchemaEntries: {e}")

    # Clean eventAttributeEntries (check for both "name" and "defaultDfeName")
    try:
        attr_entries = data["clients"]["1000"]["eventConfig"]["eventAttributeEntries"]
        kept_attr_entries = []
        for attr in attr_entries:
            # Attribute name can be in "name" or in "defaultDfeName" (use both for matching)
            names_to_check = set()
            if "name" in attr:
                names_to_check.add(attr["name"])
            if "defaultDfeName" in attr:
                names_to_check.add(attr["defaultDfeName"])
            if any(n in features_to_delete for n in names_to_check):
                count_deleted += 1
                processed_features.update(n for n in names_to_check if n in features_to_delete)
                continue
            kept_attr_entries.append(attr)
        data["clients"]["1000"]["eventConfig"]["eventAttributeEntries"] = kept_attr_entries
    except Exception as e:
        print(f"Failed eventAttributeEntries: {e}")

    # Write updated file
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)
    print("Features processed:", sorted(processed_features))
    print("Number of rows deleted:", count_deleted)


# Example usage:
clean_event_type_json(
    input_path='event.json',
    output_path='event_cleaned.json',
    features_to_delete=FEATURES_TO_DELETE
)



[ERROR] Could not load JSON due to a decoding error.
Details: Expecting property name enclosed in double quotes
Check if the JSON file is valid -- possibly keys/strings are using single quotes
or there are trailing commas or comments (which are not allowed).
Line 2, column 1 (char 2)

